# PHARMALENS AI — PROJECT 11A
## Corporate Data Integration

Purpose:
- Connect PharmaLens to corporate REST APIs.
- Provide a Salesforce connector architecture.
- Normalize corporate data into a canonical PharmaLens schema.
- Store raw and processed corporate data.
- Prepare corporate CRM, sales, orders and inventory data for the Agent.

Credentials must be stored in environment variables / deployment secrets, never in the notebook.


In [1]:
# CELL 01 — Initialization

import sys
import os
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().resolve().parents[0]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
CORPORATE_DIR = DATA_DIR / "corporate"
CORPORATE_RAW_DIR = CORPORATE_DIR / "raw"
CORPORATE_PROCESSED_DIR = CORPORATE_DIR / "processed"

for directory in [
    CORPORATE_RAW_DIR,
    CORPORATE_PROCESSED_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("PharmaLens AI — Project 11A")
print("PROJECT_ROOT:", PROJECT_ROOT)


PharmaLens AI — Project 11A
PROJECT_ROOT: D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI


In [2]:
# CELL 02 — Generic REST API

def api_get(url, headers=None, params=None, timeout=60, retries=3):
    headers = headers or {}
    last_error = None

    for attempt in range(retries):
        try:
            response = requests.get(
                url,
                headers=headers,
                params=params,
                timeout=timeout
            )
            response.raise_for_status()
            return response
        except requests.RequestException as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(2 ** attempt)

    raise last_error


def fetch_json_api(url, headers=None, params=None, timeout=60):
    return api_get(
        url,
        headers=headers,
        params=params,
        timeout=timeout
    ).json()


In [3]:
# CELL 03 — JSON → DataFrame

def normalize_column_name(column):
    column = str(column).strip()
    column = (
        column.replace(".", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace(" ", "_")
    )
    while "__" in column:
        column = column.replace("__", "_")
    return column.strip("_")


def normalize_dataframe_columns(df):
    df = df.copy()
    df.columns = [normalize_column_name(c) for c in df.columns]
    return df


def json_to_dataframe(payload, records_key=None):
    if payload is None:
        return pd.DataFrame()

    if records_key and isinstance(payload, dict):
        payload = payload.get(records_key, [])

    if isinstance(payload, dict):
        for key in ["records", "data", "results", "items"]:
            if key in payload:
                payload = payload[key]
                break

    if isinstance(payload, list):
        return pd.json_normalize(payload)

    if isinstance(payload, dict):
        return pd.json_normalize([payload])

    return pd.DataFrame()


In [4]:
# CELL 04 — Canonical PharmaLens corporate schema

CORPORATE_SCHEMA = {
    "sales": [
        "Date",
        "Product",
        "Brand",
        "Manufacturer",
        "Customer",
        "Territory",
        "Sales_Representative",
        "Units",
        "Sales_Value"
    ],
    "orders": [
        "Order_ID",
        "Order_Date",
        "Customer",
        "Product",
        "Brand",
        "Quantity",
        "Order_Value",
        "Order_Status"
    ],
    "inventory": [
        "Date",
        "Product",
        "Brand",
        "Warehouse",
        "Stock_Quantity",
        "Pending_Orders",
        "Stock_Cover_Days"
    ],
    "customers": [
        "Customer_ID",
        "Customer",
        "Customer_Type",
        "City",
        "Region",
        "Territory"
    ],
    "crm": [
        "Interaction_Date",
        "Customer",
        "HCP",
        "Sales_Representative",
        "Interaction_Type",
        "Product",
        "Brand",
        "Outcome"
    ]
}

for dataset, columns in CORPORATE_SCHEMA.items():
    print(dataset.upper(), "->", columns)


SALES -> ['Date', 'Product', 'Brand', 'Manufacturer', 'Customer', 'Territory', 'Sales_Representative', 'Units', 'Sales_Value']
ORDERS -> ['Order_ID', 'Order_Date', 'Customer', 'Product', 'Brand', 'Quantity', 'Order_Value', 'Order_Status']
INVENTORY -> ['Date', 'Product', 'Brand', 'Warehouse', 'Stock_Quantity', 'Pending_Orders', 'Stock_Cover_Days']
CUSTOMERS -> ['Customer_ID', 'Customer', 'Customer_Type', 'City', 'Region', 'Territory']
CRM -> ['Interaction_Date', 'Customer', 'HCP', 'Sales_Representative', 'Interaction_Type', 'Product', 'Brand', 'Outcome']


In [5]:
# CELL 05 — Validation

def validate_corporate_data(df, dataset_type, required_columns=None):
    df = df.copy()
    errors = []
    warnings = []

    if required_columns is None:
        required_columns = CORPORATE_SCHEMA.get(dataset_type, [])

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        errors.append(f"Missing columns: {missing}")

    if df.empty:
        warnings.append("Dataset is empty.")

    return {
        "valid": len(errors) == 0,
        "errors": errors,
        "warnings": warnings,
        "rows": len(df),
        "columns": list(df.columns),
    }


In [6]:
# CELL 06 — Salesforce configuration

SALESFORCE_CONFIG = {
    "login_url": os.getenv(
        "SALESFORCE_LOGIN_URL",
        "https://login.salesforce.com"
    ),
    "client_id": os.getenv("SALESFORCE_CLIENT_ID"),
    "client_secret": os.getenv("SALESFORCE_CLIENT_SECRET"),
    "username": os.getenv("SALESFORCE_USERNAME"),
    "password": os.getenv("SALESFORCE_PASSWORD"),
    "security_token": os.getenv("SALESFORCE_SECURITY_TOKEN"),
}

print("Salesforce environment configuration loaded.")
print("Secrets are not printed.")


Salesforce environment configuration loaded.
Secrets are not printed.


In [7]:
# CELL 07 — Salesforce connection object

class SalesforceConnection:
    def __init__(
        self,
        access_token,
        instance_url,
        api_version="v66.0"
    ):
        self.access_token = access_token
        self.instance_url = instance_url.rstrip("/")
        self.api_version = api_version

    @property
    def headers(self):
        return {
            "Authorization": f"Bearer {self.access_token}",
            "Content-Type": "application/json"
        }

    @property
    def base_url(self):
        return (
            f"{self.instance_url}/services/data/"
            f"{self.api_version}"
        )


In [8]:
# CELL 08 — Salesforce SOQL

def salesforce_query(connection, soql):
    url = f"{connection.base_url}/query"

    response = api_get(
        url=url,
        headers=connection.headers,
        params={"q": soql}
    )

    return response.json()


In [9]:
# CELL 09 — Salesforce paginated query

def salesforce_query_all(connection, soql):
    payload = salesforce_query(connection, soql)

    records = payload.get("records", [])
    next_url = payload.get("nextRecordsUrl")

    while next_url:
        response = api_get(
            connection.instance_url + next_url,
            headers=connection.headers
        )
        payload = response.json()

        records.extend(payload.get("records", []))
        next_url = payload.get("nextRecordsUrl")

    if not records:
        return pd.DataFrame()

    df = pd.json_normalize(records)

    df = df.drop(
        columns=[
            c for c in df.columns
            if c.startswith("attributes")
        ],
        errors="ignore"
    )

    return normalize_dataframe_columns(df)


In [10]:
# CELL 10 — Salesforce field mapping

SALESFORCE_MAPPING = {
    "sales": {
        "CloseDate": "Date",
        "Amount": "Sales_Value",
    },
    "orders": {
        "OrderNumber": "Order_ID",
        "EffectiveDate": "Order_Date",
        "Quantity": "Quantity",
    },
    "inventory": {
        "ProductCode": "Product",
        "QuantityOnHand": "Stock_Quantity",
    },
    "customers": {
        "Id": "Customer_ID",
        "Name": "Customer",
    },
}


def apply_field_mapping(df, mapping):
    df = df.copy()

    rename_map = {
        source: target
        for source, target in mapping.items()
        if source in df.columns
    }

    return df.rename(columns=rename_map)


In [11]:
# CELL 11 — Generic corporate API class

class CorporateAPI:
    def __init__(
        self,
        base_url,
        access_token=None,
        api_key=None
    ):
        self.base_url = base_url.rstrip("/")
        self.access_token = access_token
        self.api_key = api_key

    @property
    def headers(self):
        headers = {"Accept": "application/json"}

        if self.access_token:
            headers["Authorization"] = (
                f"Bearer {self.access_token}"
            )

        if self.api_key:
            headers["X-API-Key"] = self.api_key

        return headers

    def get(self, endpoint, params=None):
        url = (
            self.base_url
            + "/"
            + endpoint.lstrip("/")
        )

        return fetch_json_api(
            url,
            headers=self.headers,
            params=params
        )


In [12]:
# CELL 12 — Storage helpers

def save_raw_dataframe(df, dataset_name):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = (
        CORPORATE_RAW_DIR
        / f"{dataset_name}_{timestamp}.parquet"
    )
    df.to_parquet(path, index=False)
    return path


def save_processed_corporate_data(df, dataset_name):
    path = (
        CORPORATE_PROCESSED_DIR
        / f"{dataset_name}.parquet"
    )
    df.to_parquet(path, index=False)
    return path


In [13]:
# CELL 13 — Supply metrics

def calculate_supply_metrics(inventory_df):
    df = inventory_df.copy()

    for col in [
        "Stock_Quantity",
        "Pending_Orders",
        "Stock_Cover_Days"
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    if {"Stock_Quantity", "Pending_Orders"}.issubset(df.columns):
        df["Available_After_Orders"] = (
            df["Stock_Quantity"]
            - df["Pending_Orders"]
        )

    if "Stock_Cover_Days" in df.columns:
        df["Supply_Risk"] = pd.cut(
            df["Stock_Cover_Days"],
            bins=[-np.inf, 7, 30, 60, np.inf],
            labels=[
                "Critical",
                "High",
                "Moderate",
                "Healthy"
            ]
        )

    return df


In [14]:
# CELL 14 — Demo corporate sales data

demo_sales = pd.DataFrame({
    "Date": pd.date_range("2026-01-01", periods=6, freq="MS"),
    "Product": [
        "Product A", "Product A",
        "Product B", "Product B",
        "Product C", "Product C"
    ],
    "Brand": [
        "Brand A", "Brand A",
        "Brand B", "Brand B",
        "Brand C", "Brand C"
    ],
    "Manufacturer": [
        "Company A", "Company A",
        "Company B", "Company B",
        "Company C", "Company C"
    ],
    "Customer": [
        "Hospital 1", "Hospital 2",
        "Hospital 1", "Hospital 3",
        "Pharmacy 1", "Pharmacy 2"
    ],
    "Territory": [
        "Riyadh", "Riyadh",
        "Jeddah", "Jeddah",
        "Riyadh", "Dammam"
    ],
    "Sales_Representative": [
        "Rep 1", "Rep 2",
        "Rep 3", "Rep 3",
        "Rep 1", "Rep 4"
    ],
    "Units": [100, 120, 90, 110, 150, 170],
    "Sales_Value": [
        10000, 12000,
        9000, 11000,
        7500, 8500
    ]
})

display(demo_sales)


,Date,Product,Brand,Manufacturer,Customer,Territory,Sales_Representative,Units,Sales_Value
0,2026-01-01,Product A,Brand A,Company A,Hospital 1,Riyadh,Rep 1,100,10000
1,2026-02-01,Product A,Brand A,Company A,Hospital 2,Riyadh,Rep 2,120,12000
2,2026-03-01,Product B,Brand B,Company B,Hospital 1,Jeddah,Rep 3,90,9000
3,2026-04-01,Product B,Brand B,Company B,Hospital 3,Jeddah,Rep 3,110,11000
4,2026-05-01,Product C,Brand C,Company C,Pharmacy 1,Riyadh,Rep 1,150,7500
5,2026-06-01,Product C,Brand C,Company C,Pharmacy 2,Dammam,Rep 4,170,8500


In [15]:
# CELL 15 — Corporate sales KPIs

corporate_sales_kpi = (
    demo_sales
    .groupby("Brand")
    .agg(
        Sales_Value=("Sales_Value", "sum"),
        Units=("Units", "sum"),
        Customers=("Customer", "nunique"),
        Territories=("Territory", "nunique")
    )
    .reset_index()
)

corporate_sales_kpi["Average_Price"] = np.where(
    corporate_sales_kpi["Units"] > 0,
    corporate_sales_kpi["Sales_Value"]
    / corporate_sales_kpi["Units"],
    np.nan
)

display(corporate_sales_kpi)


,Brand,Sales_Value,Units,Customers,Territories,Average_Price
0,Brand A,22000,220,2,1,100.0
1,Brand B,20000,200,2,1,100.0
2,Brand C,16000,320,2,2,50.0


In [16]:
# CELL 16 — Demo inventory and supply risk

demo_inventory = pd.DataFrame({
    "Date": ["2026-08-01"] * 3,
    "Product": ["Product A", "Product B", "Product C"],
    "Brand": ["Brand A", "Brand B", "Brand C"],
    "Warehouse": ["Riyadh", "Jeddah", "Dammam"],
    "Stock_Quantity": [500, 100, 1500],
    "Pending_Orders": [200, 250, 300],
    "Stock_Cover_Days": [45, 5, 90]
})

demo_inventory = calculate_supply_metrics(
    demo_inventory
)

display(demo_inventory)


,Date,Product,Brand,Warehouse,Stock_Quantity,Pending_Orders,Stock_Cover_Days,Available_After_Orders,Supply_Risk
0,2026-08-01,Product A,Brand A,Riyadh,500,200,45,300,Moderate
1,2026-08-01,Product B,Brand B,Jeddah,100,250,5,-150,Critical
2,2026-08-01,Product C,Brand C,Dammam,1500,300,90,1200,Healthy


In [17]:
# CELL 17 — Save processed corporate datasets

sales_path = save_processed_corporate_data(
    demo_sales,
    "corporate_sales"
)

inventory_path = save_processed_corporate_data(
    demo_inventory,
    "supply_risk"
)

intelligence = (
    demo_sales
    .groupby(
        ["Brand", "Manufacturer", "Territory"]
    )
    .agg(
        Sales_Value=("Sales_Value", "sum"),
        Units=("Units", "sum"),
        Customers=("Customer", "nunique")
    )
    .reset_index()
)

intelligence["Average_Price"] = np.where(
    intelligence["Units"] > 0,
    intelligence["Sales_Value"] / intelligence["Units"],
    np.nan
)

intelligence_path = save_processed_corporate_data(
    intelligence,
    "corporate_intelligence"
)

print("Sales:", sales_path)
print("Supply:", inventory_path)
print("Intelligence:", intelligence_path)


Sales: D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI\data\corporate\processed\corporate_sales.parquet
Supply: D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI\data\corporate\processed\supply_risk.parquet
Intelligence: D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI\data\corporate\processed\corporate_intelligence.parquet


In [18]:
# CELL 18 — Agent-ready data contract

AGENT_DATASETS = {
    "corporate_sales":
        CORPORATE_PROCESSED_DIR / "corporate_sales.parquet",

    "supply_risk":
        CORPORATE_PROCESSED_DIR / "supply_risk.parquet",

    "corporate_intelligence":
        CORPORATE_PROCESSED_DIR / "corporate_intelligence.parquet"
}

for name, path in AGENT_DATASETS.items():
    print(
        name,
        "->",
        path,
        "| exists:",
        path.exists()
    )


corporate_sales -> D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI\data\corporate\processed\corporate_sales.parquet | exists: True
supply_risk -> D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI\data\corporate\processed\supply_risk.parquet | exists: True
corporate_intelligence -> D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI\data\corporate\processed\corporate_intelligence.parquet | exists: True


In [19]:
# CELL 19 — Final status

print("=" * 70)
print("PHARMALENS AI — PROJECT 11A")
print("CORPORATE DATA INTEGRATION")
print("=" * 70)
print("Generic REST API             : READY")
print("Salesforce architecture      : READY")
print("Canonical corporate schema   : READY")
print("Field mapping                : READY")
print("Pagination                   : READY")
print("Data validation              : READY")
print("Corporate storage            : READY")
print("Demand & supply framework    : READY")
print("Agent-ready datasets         : READY")
print("=" * 70)


PHARMALENS AI — PROJECT 11A
CORPORATE DATA INTEGRATION
Generic REST API             : READY
Salesforce architecture      : READY
Canonical corporate schema   : READY
Field mapping                : READY
Pagination                   : READY
Data validation              : READY
Corporate storage            : READY
Demand & supply framework    : READY
Agent-ready datasets         : READY
